# Translate MDL files to python 

@Author: Georgia 
@Last Date: 2024-05/03

This notebook is designed to translate MDL files to Python. The goal is to create a script that can automatically convert MDL files into Python code.




In [4]:
import pandas as pd
import numpy as np
import pysd as ps
import netCDF4 
import inspect
import regex as re


## Read Vensim Files

In [38]:
model = ps.read_vensim('Dataset/test.mdl', split_views=True)

/Users/gl665/PycharmProjects/Phrame-MentalMaps/new_venv/lib/python3.12/site-packages/pysd/translators/vensim/vensim_file.py:199: UserWarning: Only a single view with no subviews was detected. The model will be built in a single file.
  warnings.warn(
/Users/gl665/PycharmProjects/Phrame-MentalMaps/new_venv/lib/python3.12/site-packages/pysd/builders/python/python_expressions_builder.py:2281: SyntaxWarning: 'chicken' has no equation specified
  return builder.build(arguments)
/Users/gl665/PycharmProjects/Phrame-MentalMaps/new_venv/lib/python3.12/site-packages/pysd/builders/python/python_expressions_builder.py:2281: SyntaxWarning: 'egg' has no equation specified
  return builder.build(arguments)
/Users/gl665/PycharmProjects/Phrame-MentalMaps/new_venv/lib/python3.12/site-packages/pysd/builders/python/python_expressions_builder.py:2281: SyntaxWarning: 'food' has no equation specified
  return builder.build(arguments)
/Users/gl665/PycharmProjects/Phrame-MentalMaps/new_venv/lib/python3.12/site

In [39]:
model.dependencies

{'final_time': {},
 'initial_time': {},
 'saveper': {'time_step': 1},
 'time_step': {},
 'chicken': {'food': 1},
 'egg': {'chicken': 1},
 'food': {'water': 1},
 'water': {'egg': 1, 'food': 1}}

In [24]:
dir(model)

['_Macro__get_varname_from_ext_name',
 '_Macro__include_for_serialization',
 '__call__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_add_constant_cache',
 '_assign_cache',
 '_assign_cache_type',
 '_build_doc',
 '_components_setter_tracker',
 '_config_simulation',
 '_constant_component',
 '_constant_funcs',
 '_count_calls',
 '_data_elements',
 '_default_return_columns',
 '_dependencies',
 '_doc',
 '_dynamicstateful_elements',
 '_euler_step',
 '_external_elements',
 '_get_data',
 '_get_elements_to_initialize',
 '_get_full_dependencies',
 '_get_initialize_order',
 '_integrate',
 '_integrate_step',
 '_isdynamic',
 '_lookup_elements',


In [41]:
model.get_dependencies()

Dependencies(c_vars={'time_step', 'initial_time', 'saveper', 'time', 'final_time'}, d_deps={'initial': set(), 'step': set(), 'lookup': set()}, s_deps=set())

# Read txt file of MDL file

In [17]:
with open('Dataset/test.mdl', 'r') as f:
    lines = f.readlines()
    # print(lines)
    
# Compile regex to match line with sketch info    

start_pattern = re.compile(r"\\\---///*") 
end_pattern = re.compile(r'\/{3}-+\\{3}')
                         
extracted = []
extract = False

for line in lines:
    if end_pattern.search(line):
        # Stop extracting if end pattern found
        extract = False

    elif start_pattern.search(line):
        # Start new extraction
        extracted = []
        extract = True

    elif extract:
        # Append to extracted lines
        extracted.append(line)

extracted = extracted[3:]
print(extracted)


['10,1,egg,276,168,11,8,8,3,0,0,0,0,0,0,0,0,0,0,0,0\n', '10,2,water,260,253,16,8,8,3,0,0,0,0,0,0,0,0,0,0,0,0\n', '10,3,food,255,380,13,8,8,3,0,0,0,0,0,0,0,0,0,0,0,0\n', '1,4,1,2,1,0,45,0,2,64,0,-1--1--1,|12||0-0-0,1|(265,199)|\n', '1,5,2,3,0,0,43,0,2,64,0,-1--1--1,|12||0-0-0,1|(257,309)|\n', '1,6,3,2,1,0,43,0,2,64,0,-1--1--1,|12||0-0-0,1|(333,285)|\n', '10,7,chicken,388,145,21,8,8,3,0,0,0,0,0,0,0,0,0,0,0,0\n', '1,8,2,7,1,0,0,0,0,64,0,-1--1--1,,1|(344,196)|\n', '1,9,7,1,1,0,0,0,0,64,0,-1--1--1,,1|(315,136)|\n']


In [16]:
df = pd.DataFrame(extracted)
df = df[0].str.split(',', expand = True)
# change column 0 dytpe to int
df[0] = df[0].astype(int)
# Turn this into a dataframe by seperating each cell with ,
df

,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,10,1,egg,276,168,11,8,8,3,0,...,0,0,0,0,0,0,0,0,0,0\n
1,10,2,water,260,253,16,8,8,3,0,...,0,0,0,0,0,0,0,0,0,0\n
2,10,3,food,255,380,13,8,8,3,0,...,0,0,0,0,0,0,0,0,0,0\n
3,1,4,1,2,1,0,45,0,2,64,...,-1--1--1,|12||0-0-0,1|(265,199)|\n,None,None,None,None,None,None
4,1,5,2,3,0,0,43,0,2,64,...,-1--1--1,|12||0-0-0,1|(257,309)|\n,None,None,None,None,None,None
5,1,6,3,2,1,0,43,0,2,64,...,-1--1--1,|12||0-0-0,1|(333,285)|\n,None,None,None,None,None,None
6,10,7,chicken,388,145,21,8,8,3,0,...,0,0,0,0,0,0,0,0,0,0\n
7,1,8,2,7,1,0,0,0,0,64,...,-1--1--1,,1|(344,196)|\n,None,None,None,None,None,None
8,1,9,7,1,1,0,0,0,0,64,...,-1--1--1,,1|(315,136)|\n,None,None,None,None,None,None


In [7]:
# check datatype of df 
# for cols in df.columns:
#     print(df[cols].dtype)
#     

43 is positive, 45 is negative 


In [18]:
# Grab dependencies 
df_links = df.loc[df[0]==1,  ]
df_links = df_links[[0,1, 2,3,6]]
df_links = df_links.rename(columns={0: 'variable type', 1: 'index',2: 'from', 3: 'to', 6: 'polarity'})
df_links['polarity'] = df_links['polarity'].astype(int)
df_links['links'] = 1
# Give dataframe new column names
# If == 43 transform to 1, if == 45 transform to -1, else transform to 0
# if df_links['polarity'] == 43:
#     df_links['polarity_type'] = 1
# elif df_links['polarity'] == 45:
#     df_links['polarity_tupe'] = -1
# else: 
#     df_links['polarity_type'] = 0
# 

df_links['polarity_type'] = np.where(df_links['polarity'] == 43, 1,
                                     np.where(df_links['polarity'] == 45, -1, 3))

    
# df_links['polarity_type'] = np.where(df_links['polarity'] == 43, 1, -1)
df_links

,variable type,index,from,to,polarity,links,polarity_type
3,1,4,1,2,45,1,-1
4,1,5,2,3,43,1,1
5,1,6,3,2,43,1,1
7,1,8,2,7,0,1,3
8,1,9,7,1,0,1,3


In [13]:
df_vars = df.loc[df[0]==10,  ]
# df_vars
# Create a dictionary based on  dataframe column 1 and 2
dict_var = df_vars.set_index(1)[2].to_dict()
dict_var


{'1': 'egg', '2': 'water', '3': 'food', '7': 'chicken'}

In [14]:
# Make a adjency matrix from df_links column from, to and links 
link_matrix = df_links.pivot(index='from', columns='to', values='links').fillna(0).astype(int)

# change the columns names by using dict_var
link_matrix.columns = link_matrix.columns.map(dict_var)
link_matrix

to,egg,water,food,chicken
from,,,,
1,0,1,0,0
2,0,0,1,1
3,0,1,0,0
7,1,0,0,0


In [15]:
polarity_matrix = df_links.pivot(index='from', columns = 'to', values = 'polarity_type').fillna(0).astype(int)
polarity_matrix.columns = polarity_matrix.columns.map(dict_var)
polarity_matrix

to,egg,water,food,chicken
from,,,,
1,0,-1,0,0
2,0,0,1,3
3,0,1,0,0
7,3,0,0,0
